**Link al repositorio de GitHub:** [https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo](https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo)
Carlos Alberto Valladares 221164

# CC3104 - Aprendizaje por Refuerzo
## Laboratorio 2 - Entrega Parcial

---

### Task 1: Diseño formal del MDP

#### 1. Espacio de estados $\mathcal{S}$
Para modelar el sistema de ascensores del hospital (5 pisos), el estado $s \in \mathcal{S}$ debe capturar la ubicación del ascensor y las solicitudes pendientes. Lo definimos como una tupla: $s = (e, d_1, d_2, d_3, d_4, d_5)$:
*   **$e \in \{1, 2, 3, 4, 5\}$:** Piso actual del ascensor. *Justificación:* Necesario para calcular distancias y movimientos válidos.
*   **$d_i \in \{0, 1, 2\}$:** Estado de demanda en el piso $i$. (0 = sin demanda, 1 = normal, 2 = emergencia). *Justificación:* Fundamental para decidir el próximo destino y priorizar emergencias médicas.
*   **Variables omitidas:** Tiempo de espera de cada paciente, dirección de viaje deseada por el paciente, y capacidad de peso del ascensor. 
*   **Supuesto y Consecuencia:** Asumimos capacidad infinita y que minimizar la distancia a las llamadas minimiza indirectamente el tiempo de espera. La consecuencia de omitir el tiempo de espera explícito es que el ascensor podría sufrir de "inanición" (starvation): si siguen apareciendo emergencias en el piso 5, un paciente normal en el piso 1 podría esperar infinitamente porque el estado no refleja su urgencia acumulada.

#### 2. Espacio de acciones $\mathcal{A}$
El espacio de acciones es **discreto**, dado que el ascensor se mueve entre pisos enteros.
*   $\mathcal{A} = \{\text{Subir, Bajar, Permanecer}\}$
*   **Restricciones:** En el piso $e=5$, la acción "Subir" está restringida. En el piso $e=1$, la acción "Bajar" está restringida.
*   **Modelado en el MDP:** Se modela restringiendo el conjunto de acciones válidas según el estado (es decir, el conjunto de acciones depende del estado, $\mathcal{A}(s)$). Alternativamente, se modela en la función de transición: si se elige "Subir" en el piso 5, $P(s'|s, a)$ devuelve el mismo piso 5 con probabilidad 1, y la función de recompensa aplica una leve penalización por un comando inválido.

#### 3. Función de recompensa $R(s, a, s')$
El objetivo es minimizar esperas, priorizar emergencias y ahorrar energía:
*   **Diseño:** $R = -1$ por cada paso de tiempo (penaliza espera y energía). $+10$ al atender (llegar a un piso con $d_i=1$) una demanda normal. $+50$ al atender una emergencia ($d_i=2$). $-0.5$ adicional por moverse si no hay demandas en ningún piso (para ahorrar energía).
*   **Ponderación:** Se priorizan las emergencias ($+50$) por sobre las llamadas normales ($+10$), compensando el costo de viaje ($-1$/piso) de cruzar todo el edificio para salvar una vida.
*   **Consecuencias de mala ponderación:** Si la recompensa por emergencia fuera muy baja (ej. $+12$), el ascensor preferiría atender a dos pacientes normales cercanos en lugar de viajar al 5to piso por una emergencia. Si el castigo por gastar energía fuera muy alto, el ascensor se negaría a moverse a menos que la emergencia ocurra en su mismo piso.

#### 4. Función de transición $P(s' | s, a)$
El entorno es **estocástico**. Aunque el movimiento mecánico del ascensor es altamente determinista, la **demanda es aleatoria**. Las fuentes de aleatoriedad son los pacientes y médicos presionando los botones en tiempos impredecibles.
Ejemplos de transiciones (asumiendo que moverse toma 1 paso y que hay probabilidad de que aparezcan nuevas demandas):
1.  **Movimiento normal sin nuevas demandas:** $P(e=2, d_3=1 \mid e=1, d_3=1, a=\text{Subir}) = 0.8$. (El ascensor sube exitosamente y el sistema no recibe nuevas llamadas).
2.  **Movimiento con aparición de emergencia:** $P(e=2, d_3=1, d_5=2 \mid e=1, d_3=1, a=\text{Subir}) = 0.05$. (Mientras subía al piso 2, alguien en el 5 presionó el botón de emergencia).
3.  **Llegada y resolución:** $P(e=3, d_3=0 \mid e=2, d_3=1, a=\text{Subir}) = 0.9$. (Al llegar al piso 3, la demanda $d_3$ se despeja o resuelve automáticamente).
4.  **Inactividad y nueva llamada:** $P(e=1, d_2=1 \mid e=1, \text{sin demandas}, a=\text{Permanecer}) = 0.1$. (El ascensor está esperando y aparece un paciente normal en el piso 2).

#### 5. Factor de descuento $\gamma$
Se propone **$\gamma = 0.95$**.
*   **Justificación:** En un hospital, nos interesa resolver las necesidades rápidamente, por lo que el futuro a muy largo plazo es incierto y menos relevante que los pacientes actuales. 
*   **Cambio cualitativo de la política:**
    *   Si $\gamma \approx 0$ (Miope): El ascensor solo verá recompensas a 1 paso de distancia. Solo atenderá solicitudes si casualmente está en el piso adyacente, negándose a viajar 3 pisos por una emergencia porque la recompensa descontada sería $0^3 \times 50 = 0$.
    *   Si $\gamma \approx 1$ (Visión lejana): El ascensor valorará las recompensas distantes casi igual que las inmediatas. Estará dispuesto a cruzar todo el edificio para apagar emergencias sabiendo que el valor matemático no decaerá.

---

### Task 2: Preguntas Teóricas

#### 1. Número de políticas deterministas posibles
*   **Cálculo del Espacio de Estados $|S|$:** Hay 5 pisos para el ascensor. Para los 5 pisos, cada uno tiene 3 estados de demanda (0, 1, 2). Total de estados = $5 \times 3^5 = 5 \times 243 = 1,215$ estados.
*   **Cálculo de Políticas:** Para cada estado, hay $|A| = 3$ acciones posibles. El total de políticas deterministas únicas es $|A|^{|S|} = 3^{1215}$.
*   **Inviabilidad:** El número $3^{1215}$ es astronómicamente inmenso (mucho mayor a la cantidad de átomos en el universo observable). Es completamente imposible usar fuerza bruta o enumeración exhaustiva para evaluar cada política y encontrar la óptima, demostrando por qué necesitamos algoritmos como Value Iteration o Policy Iteration.

#### 2. Iteraciones teóricas de Policy Evaluation
Buscamos reducir el error a $\theta = 0.01$. Dado que $T^\pi$ es una contracción de factor $\gamma = 0.95$, y asumiendo la peor recompensa máxima posible $R_{max} = 50$, el error inicial está acotado por:
$$ \|V_0 - V^\pi\|_\infty \leq \frac{R_{max}}{1-\gamma} = \frac{50}{1 - 0.95} = 1000 $$
La cota geométrica de convergencia nos dice que en el paso $k$:
$$ \|V_k - V^\pi\|_\infty \leq \gamma^k \|V_0 - V^\pi\|_\infty $$
Sustituyendo los valores para encontrar $k$:
$$ 0.95^k \times 1000 < 0.01 $$
$$ 0.95^k < \frac{0.01}{1000} = 0.00001 $$
Aplicando logaritmo natural en ambos lados:
$$ k \ln(0.95) < \ln(0.00001) $$
$$ k(-0.05129) < -11.5129 $$
$$ k > \frac{-11.5129}{-0.05129} \approx 224.45 $$
**Conclusión:** Se necesitarían teóricamente al menos **225 iteraciones** de Policy Evaluation para garantizar que el error máximo baje de 0.01 en el peor de los casos.

#### 3. Policy Iteration vs Value Iteration (Tiempo de Cómputo)
*   **Anticipación:** Se anticipa que **Value Iteration convergerá más rápido** en términos de tiempo de cómputo total para este MDP.
*   **Justificación:**
    *   *Tamaño del estado ($1215$):* En Policy Iteration, cada paso de evaluación requiere resolver un sistema lineal de $1215 \times 1215$ o hacer $\approx 225$ iteraciones iterativas de evaluación *por cada* mejora de política. 
    *   *Número de acciones (3):* Dado que solo hay 3 acciones posibles, el costo de calcular el `max` sobre las acciones en Value Iteration es computacionalmente despreciable (muy barato).
    *   *Trade-off:* Value Iteration trunca la fase de evaluación a un solo paso de barrido y combina la mejora. Aunque matemáticamente podría requerir más pasos totales (iteraciones externas) que Policy Iteration, el tiempo real de cada paso es minúsculo en comparación con la evaluación exhaustiva de PI. Por ende, para un MDP de este tamaño, el costo por iteración de VI es el ganador absoluto.